# Graded Challenge 6

Nama  : Putri Joeliya

Batch : CODA-RMT-020

Link  : https://docs.google.com/presentation/d/1MPMYykM4spd1AahWRVYrg4HXaxqnvaL7MbHcMjfeMGY/edit?usp=sharing

Program ini dibuat untuk melakukan transformasi terhadap data.

## Import PySpark
This code initializes a PySpark session and loads the PostgreSQL JDBC driver so PySpark can connect and transfer data to/from PostgreSQL database

In [1]:
import pyspark 

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WriteToPostgres") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

spark = SparkSession.builder.getOrCreate()
spark

## Loading CSV File & Inspecting Data 

this steps are use to performs Data Ingestion and Data Inspection

### loading Dataset From CSV 

In [3]:
df_users_raw = spark.read.csv('dim_users.csv', header=True, inferSchema=True)
df_products_raw = spark.read.csv('dim_products.csv', header=True, inferSchema=True)
df_orders_raw = spark.read.csv('dim_orders.csv', header=True, inferSchema=True)
df_fact_raw = spark.read.csv('fact_sales.csv', header=True, inferSchema=True)

### Checking Rows 

In [4]:
print('dim_users')
df_users_raw.show(5)

print('dim_products')
df_products_raw.show(5)

print('dim_orders')
df_orders_raw.show(5)

print('fact_sales')
df_fact_raw.show(5)

dim_users
+-----+----------+---------+--------------------+---+------+-----+----+-------+--------------+
|   id|first_name|last_name|               email|age|gender|state|city|country|traffic_source|
+-----+----------+---------+--------------------+---+------+-----+----+-------+--------------+
|45243| Catherine|  Jackson|catherinejackson@...| 58|     F| Acre|null| Brasil|        Search|
|31812|    Steven|Zimmerman|stevenzimmerman@e...| 21|     M| Acre|null| Brasil|        Search|
|11963|     Scott|   Flores|scottflores@examp...| 18|     M| Acre|null| Brasil|        Search|
|13073|   Stanley|    Evans|stanleyevans@exam...| 60|     M| Acre|null| Brasil|        Search|
|69983|  Patricia|    Smith|patriciasmith@exa...| 14|     F| Acre|null| Brasil|        Search|
+-----+----------+---------+--------------------+---+------+-----+----+-------+--------------+
only showing top 5 rows
dim_products
+----------+-----------+--------------------+-----+------------------+----------+
|product_id|   c

### Checking Dtypes

In [5]:
print('dim_users')
df_users_raw.printSchema(5)

print('dim_products')
df_products_raw.printSchema(5)

print('dim_orders')
df_orders_raw.printSchema(5)

print('fact_sales')
df_fact_raw.printSchema(5)

dim_users
root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_source: string (nullable = true)

dim_products
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- retail_price: double (nullable = true)
 |-- department: string (nullable = true)

dim_orders
root
 |-- order_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- num_of_item: integer (nullable = true)

fact_sales
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- date_i

### Preforming EDA

In [6]:
print('dim_users')
df_users_raw.describe().show(5)

print('dim_products')
df_products_raw.describe().show(5)

print('dim_orders')
df_orders_raw.describe().show(5)

print('fact_sales')
df_fact_raw.describe().show(5)

dim_users
+-------+------------------+----------+---------+--------------------+----------------+------+-------------+------+-------------+--------------+
|summary|                id|first_name|last_name|               email|             age|gender|        state|  city|      country|traffic_source|
+-------+------------------+----------+---------+--------------------+----------------+------+-------------+------+-------------+--------------+
|  count|            100000|    100000|   100000|              100000|          100000|100000|       100000|100000|       100000|        100000|
|   mean|           50000.5|      NULL|     NULL|                NULL|        40.94326|  NULL|         NULL|  NULL|         NULL|          NULL|
| stddev|28867.657796687763|      NULL|     NULL|                NULL|16.9911555688857|  NULL|         NULL|  NULL|         NULL|          NULL|
|    min|                 1|     Aaron|   Abbott|aaronallen@exampl...|              12|     F|         Acre|Aachen|    A

## Data Transfom

### a. User Table (dim_users)

* checking and removing duplicate
* handling missing value for null 

In [7]:
from pyspark.sql.functions import col, when

df_users = df_users_raw.dropDuplicates(['id']) \
    .withColumn("city", when(col("city").isNull() | (col("city") == "null") | (col("city") == ""), "Unknown").otherwise(col("city"))) \
    .withColumn("state", when(col("state").isNull() | (col("state") == "null") | (col("state") == ""), "Unknown").otherwise(col("state"))) \
    .withColumnRenamed("id", "user_id")


### b. Products table (dim_products)

   * Make sure no duplicate on dim_products
   * Delete the rows with Null data in column "name"
   * Fill the null value with 'Unknown' in column "brand"

In [8]:
df_products = df_products_raw.dropDuplicates(['product_id'])
df_products = df_products.dropna(subset=["name"]).fillna({"brand": "Unknown"})
df_products.show(5)

+----------+-----------+--------------------+------------------+------------+----------+
|product_id|   category|                name|             brand|retail_price|department|
+----------+-----------+--------------------+------------------+------------+----------+
|         1|Tops & Tees|Seven7 Women's Lo...|            Seven7|        49.0|     Women|
|         3|Tops & Tees|Calvin Klein Jean...|Calvin Klein Jeans|        69.5|     Women|
|         5|Tops & Tees|Anne Klein Women'...|        Anne Klein|        94.0|     Women|
|         6|Tops & Tees|Wilt Women's Colo...|              Wilt|       132.0|     Women|
|        12|Tops & Tees|Jones New York Wo...|    Jones New York|        99.0|     Women|
+----------+-----------+--------------------+------------------+------------+----------+
only showing top 5 rows


In [9]:
df_products.count()

29118

### c. Order Table (dim_orders)
* Make sure no duplicate on dim_orders

In [10]:
df_orders = df_orders_raw.dropDuplicates(['order_id'])
df_orders.show(5)

+--------+---------+-------------------+-----------+
|order_id|   status|         created_at|num_of_item|
+--------+---------+-------------------+-----------+
|       1|Cancelled|2020-10-03 13:54:24|          1|
|       3| Complete|2025-04-30 22:07:05|          1|
|       5| Complete|2026-03-29 20:09:14|          1|
|       6|  Shipped|2026-05-26 00:36:04|          1|
|      12| Complete|2023-02-14 10:02:00|          1|
+--------+---------+-------------------+-----------+
only showing top 5 rows


In [11]:
df_orders.count()

125153

### d. Date Table (dim_date)
* Make a new table for dataframe dim_date 

In [12]:
from pyspark.sql.functions import (
    col, 
    dayofmonth, 
    month, 
    date_format, 
    concat, 
    lit, 
    quarter, 
    year
)
df_date = (
    df_fact_raw
        .select('date_id').distinct()
        .withColumn('full_date', col('date_id'))
        .withColumn('day', dayofmonth(col('date_id')))
        .withColumn('month', month(col('date_id')))
        .withColumn('month_name', date_format(col('date_id'), 'MMMM'))
        .withColumn('quarter', concat(lit('Q'), quarter(col('date_id'))))
        .withColumn('year', year(col('date_id')))
        .withColumn('day_of_week', date_format(col('date_id'), 'EEEE'))
)

In [13]:
df_date.show(5)

+----------+----------+---+-----+----------+-------+----+-----------+
|   date_id| full_date|day|month|month_name|quarter|year|day_of_week|
+----------+----------+---+-----+----------+-------+----+-----------+
|2025-10-21|2025-10-21| 21|   10|   October|     Q4|2025|    Tuesday|
|2026-02-13|2026-02-13| 13|    2|  February|     Q1|2026|     Friday|
|2024-09-18|2024-09-18| 18|    9| September|     Q3|2024|  Wednesday|
|2023-07-15|2023-07-15| 15|    7|      July|     Q3|2023|   Saturday|
|2020-08-24|2020-08-24| 24|    8|    August|     Q3|2020|     Monday|
+----------+----------+---+-----+----------+-------+----+-----------+
only showing top 5 rows


In [14]:
df_date.printSchema()

root
 |-- date_id: date (nullable = true)
 |-- full_date: date (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- quarter: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_of_week: string (nullable = true)



### e. Sales Table (fact_sales)
* removing row from 'null'
* clean duplicate


In [15]:
df_fact_sales = df_fact_raw.dropna(subset=["order_id", "product_id", "user_id", "date_id", "total_sales", "total_cost"])
df_fact_sales = df_fact_sales.dropDuplicates(subset=["order_id", "product_id"])
df_fact_sales.show(5)

+--------+----------+-------+----------+--------+-----------+----------+------------+
|order_id|product_id|user_id|   date_id|quantity|total_sales|total_cost|total_profit|
+--------+----------+-------+----------+--------+-----------+----------+------------+
|   77831|     14159|  62418|2022-04-06|       1|       0.49|      0.18|        0.31|
|   82555|     28913|  66198|2026-06-24|       1|       2.59|      1.07|        1.52|
|  123392|     24922|  98914|2026-08-04|       1|        4.5|      2.76|        1.74|
|   68055|     10690|  54615|2025-06-11|       1|       4.95|      2.61|        2.34|
|   54381|      9001|  43657|2025-06-29|       1|       5.99|      2.25|        3.74|
+--------+----------+-------+----------+--------+-----------+----------+------------+
only showing top 5 rows


## Data Loading To PostgreSQL 

In [16]:
# filtering data because some of product id is invalid in fact_sales
df_fact_sales = df_fact_sales.join(
    df_products.select("product_id"), 
    on="product_id", 
    how="inner"
)
print(f"Total : {df_fact_sales.count():,}")

Total : 181,115


In [17]:

postgres_url = "jdbc:postgresql://host.docker.internal:5432/p2g6_putri_joeliya_loading"
postgres_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}


df_users.write.jdbc(url=postgres_url, table="dim_users", mode="append", properties=postgres_properties)
df_products.write.jdbc(url=postgres_url, table="dim_products", mode="append", properties=postgres_properties)
df_orders.write.jdbc(url=postgres_url, table="dim_orders", mode="append", properties=postgres_properties)
df_date.write.jdbc(url=postgres_url, table="dim_date", mode="append", properties=postgres_properties)
df_fact_sales.write.jdbc(url=postgres_url, table="fact_sales", mode="append", properties=postgres_properties)

